<a href="https://colab.research.google.com/github/mbmanoj904-hue/Generative-ai/blob/main/SBI_Bank_Chat_Bot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# STEP 1: INSTALL REQUIRED LIBRARIES
# ============================================================

!pip install -q langchain-community langchain-huggingface \
langchain-text-splitters faiss-cpu pypdf beautifulsoup4 \
langchain-core openai

import os
from openai import OpenAI
from google.colab import userdata

# LangChain Imports
from langchain_core.documents import Document
from langchain_community.document_loaders import WebBaseLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS


# ============================================================
# STEP 2: INGEST SBI DATA
# ============================================================

print("Step 1: Ingesting SBI banking data...")

sbi_data = [

    "State Bank of India (SBI) is one of the largest public sector banks in India.",

    """SBI provides banking services such as savings accounts,
    current accounts, fixed deposits, loans, credit cards,
    internet banking and mobile banking.""",

    """SBI savings accounts allow customers to deposit money,
    withdraw money, transfer funds and use digital banking services.""",

    """SBI provides home loans to eligible customers for purchasing,
    constructing or renovating residential properties.""",

    """SBI provides education loans to eligible students for
    higher education in India and abroad, subject to eligibility
    conditions.""",

    """SBI provides personal loans to eligible customers.
    Loan approval, interest rate and loan amount depend on
    applicable SBI rules.""",

    """SBI Internet Banking allows customers to check account
    balances, transfer money, pay bills and access other
    banking services online.""",

    """SBI YONO is SBI's digital banking platform that provides
    banking and financial services through a mobile application.""",

    """Customers should never share their ATM PIN, debit card PIN,
    OTP, internet banking password or other confidential banking
    credentials with anyone.""",

    """For banking complaints or assistance, customers should use
    official SBI customer-service channels."""
]


# Convert text into LangChain Documents

text_documents = [
    Document(
        page_content=text,
        metadata={"source": "SBI Banking Information"}
    )
    for text in sbi_data
]


# ============================================================
# STEP 3: LOAD SBI WEBSITE
# ============================================================

urls = [
    "https://sbi.co.in/"
]

try:

    web_loader = WebBaseLoader(urls)

    web_documents = web_loader.load()

    print("SBI website loaded successfully.")

except Exception as e:

    print("Warning: SBI website could not be loaded.")

    web_documents = []


# ============================================================
# STEP 4: LOAD SBI PDF
# ============================================================

pdf_path = "sbi_brochure.pdf"

try:

    pdf_loader = PyPDFLoader(pdf_path)

    pdf_documents = pdf_loader.load()

    print("SBI PDF loaded successfully.")

except Exception:

    print(
        "Warning: PDF 'sbi_brochure.pdf' not found. "
        "Skipping PDF ingestion."
    )

    pdf_documents = []


# ============================================================
# STEP 5: MERGE ALL SOURCES
# ============================================================

all_documents = (
    text_documents +
    web_documents +
    pdf_documents
)

print(
    f"Total raw pages/documents loaded: {len(all_documents)}"
)


# ============================================================
# STEP 6: DATA CHUNKING
# ============================================================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(
    all_documents
)

print(
    f"Data chopped into {len(chunks)} searchable chunks."
)


# ============================================================
# STEP 7: HUGGINGFACE EMBEDDINGS + FAISS
# ============================================================

print("Building FAISS Vector Database...")

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

vector_db = FAISS.from_documents(
    chunks,
    embeddings
)

print("FAISS database created successfully.")


# ============================================================
# STEP 8: GROQ API
# ============================================================

groq_api_key = userdata.get("Groq")

if not groq_api_key:

    raise ValueError(
        "Groq API key not found. "
        "Add a secret named 'Groq' in Google Colab."
    )


client = OpenAI(
    api_key=groq_api_key,
    base_url="https://api.groq.com/openai/v1"
)


# ============================================================
# STEP 9: START SBI CHATBOT
# ============================================================

print("\n🏦 SBI Banking Assistant Online!")
print("Ask about SBI accounts, loans, Internet Banking, YONO, etc.")
print("Type 'exit' to stop.")
print("-" * 60)


# ============================================================
# STEP 10: CHAT LOOP
# ============================================================

while True:

    user_query = input("\nYou: ")

    if user_query.lower().strip() == "exit":

        print("SBI Assistant: Thank you!")
        break


    # --------------------------------------------------------
    # SEARCH FAISS
    # --------------------------------------------------------

    relevant_chunks = vector_db.similarity_search(
        user_query,
        k=3
    )


    # Combine retrieved information

    retrieved_context = "\n\n".join(
        [
            doc.page_content
            for doc in relevant_chunks
        ]
    )


    # --------------------------------------------------------
    # SYSTEM PROMPT
    # --------------------------------------------------------

    system_prompt = f"""
You are an SBI Banking Assistant.

Answer the user's question using ONLY the information
provided in the CONTEXT.

Rules:

1. Do not guess.
2. Do not invent SBI policies.
3. Do not invent interest rates or charges.
4. If the answer is not available in the context,
   say exactly:

"I do not have enough information."

5. Give simple and clear answers.
6. Never ask the user for OTP, PIN, password,
   CVV or other confidential banking information.

CONTEXT:

{retrieved_context}
"""


    # --------------------------------------------------------
    # CALL GROQ
    # --------------------------------------------------------

    response = client.chat.completions.create(

        # IMPORTANT:
        # No space before the model name
        model="openai/gpt-oss-120b",

        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_query
            }
        ],

        temperature=0
    )


    # --------------------------------------------------------
    # DISPLAY ANSWER
    # --------------------------------------------------------

    answer = response.choices[0].message.content

    print("\nSBI AI:", answer)

Step 1: Ingesting SBI banking data...
SBI website loaded successfully.
Total raw pages/documents loaded: 11
Data chopped into 11 searchable chunks.
Building FAISS Vector Database...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

FAISS database created successfully.

🏦 SBI Banking Assistant Online!
Ask about SBI accounts, loans, Internet Banking, YONO, etc.
Type 'exit' to stop.
------------------------------------------------------------

You: sbi stand for

SBI AI: SBI stands for **State Bank of India**.

You: sbi provides the online transaction

SBI AI: Yes. SBI’s Internet Banking lets you perform online transactions such as checking balances, transferring money and paying bills.

You: sbi provides the banking servies? what are

SBI AI: SBI provides the following banking services:

- Savings accounts  
- Current accounts  
- Fixed deposits  
- Loans  
- Credit cards  
- Internet banking  
- Mobile banking
